# Proyecto de Recuperación de Información


## Análisis y Procesamiento de Corpus de Documentos
- Carga de datos
- Limpieza y exploración
- Preprocesamiento (tokenización, eliminación de stopwords, normalización, stemming)


In [1]:
# Instalación de dependencias
#!pip install nltk scikit-learn numpy pandas kagglehub

### Sección 1: Configuración e Instalación


In [2]:
from pathlib import Path
import pandas as pd
import unicodedata
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [3]:
# Descargar recursos de NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

True

### Sección 2: Carga de Datos


In [4]:
# Descarga del dataset (comentado si ya existe)
import kagglehub

# Download latest version into ./data
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)
path = kagglehub.dataset_download(
    "thedevastator/uncovering-financial-insights-with-the-reuters-2",
    output_dir=str(data_dir),
)

print("Path to dataset files:", path)

C:\Users\Erick\Documents\U\7\Recuperacion de informacion\Proyecto 01\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: data


In [5]:
for file in data_dir.iterdir():
    print(f"  - {file.name}")


  - .complete
  - ModApte_test.csv
  - ModApte_train.csv
  - ModApte_unused.csv
  - ModHayes_test.csv
  - ModHayes_train.csv
  - ModLewis_test.csv
  - ModLewis_train.csv
  - ModLewis_unused.csv


In [6]:
# Seleccionar dataset de trabajo
work_file = "data/ModApte_train.csv"
print(f"\nDataset seleccionado: {work_file}")


Dataset seleccionado: data/ModApte_train.csv


In [7]:
def load_dataset(dataset):
    """
    Carga un archivo CSV y muestra información básica del dataset.

    Args:
        dataset (str): Ruta del archivo CSV a cargar

    Returns:
        pd.DataFrame: DataFrame con los datos cargados
    """
    df = pd.read_csv(dataset)

    print("=" * 100)
    print(f"Dimensión del dataset: {df.shape}")
    print(f"Columnas del dataset:\n{list(df.columns)}")
    print("=" * 100)

    # Revisar nulos
    print("\nValores nulos:")
    print(df.isnull().sum())

    print("\nPrimeras filas:")
    print(df.head(10))

    return df


### Sección 3: Exploración de Datos

In [8]:
corpus = load_dataset(work_file)

Dimensión del dataset: (9603, 13)
Columnas del dataset:
['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title']

Valores nulos:
text           787
text_type        0
topics           0
lewis_split      0
cgis_split       0
old_id           0
new_id           0
places           0
people           0
orgs             0
exchanges        0
date             0
title           54
dtype: int64

Primeras filas:
                                                text text_type  \
0  Showers continued throughout the week in\r\nth...    "NORM"   
1  The U.S. Agriculture Department\r\nreported th...    "NORM"   
2  Argentine grain board figures show\r\ncrop reg...    "NORM"   
3  Moody's Investors Service Inc said it\r\nlower...    "NORM"   
4  Champion Products Inc said its\r\nboard of dir...    "NORM"   
5  Computer Terminal Systems Inc said\r\nit has c...    "NORM"   
6  Shr 34 cts vs 1.19 dlrs\r\n    Net 807,000 vs .

In [9]:
print("Vista previa de los datos:")
corpus.head(10)

Vista previa de los datos:


,text,text_type,topics,lewis_split,cgis_split,old_id,new_id,places,people,orgs,exchanges,date,title
0,Showers continued throughout the week in\r\nth...,"""NORM""",['cocoa'],"""TRAIN""","""TRAINING-SET""","""5544""","""1""",['el-salvador' 'usa' 'uruguay'],[],[],[],26-FEB-1987 15:01:01.79,BAHIA COCOA REVIEW
1,The U.S. Agriculture Department\r\nreported th...,"""NORM""",['grain' 'wheat' 'corn' 'barley' 'oat' 'sorghum'],"""TRAIN""","""TRAINING-SET""","""5548""","""5""",['usa'],[],[],[],26-FEB-1987 15:10:44.60,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE
2,Argentine grain board figures show\r\ncrop reg...,"""NORM""",['veg-oil' 'linseed' 'lin-oil' 'soy-oil' 'sun-...,"""TRAIN""","""TRAINING-SET""","""5549""","""6""",['argentina'],[],[],[],26-FEB-1987 15:14:36.41,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS
3,Moody's Investors Service Inc said it\r\nlower...,"""NORM""",[],"""TRAIN""","""TRAINING-SET""","""5551""","""8""",['usa'],[],[],[],26-FEB-1987 15:15:40.12,USX &lt;X> DEBT DOWGRADED BY MOODY'S
4,Champion Products Inc said its\r\nboard of dir...,"""NORM""",['earn'],"""TRAIN""","""TRAINING-SET""","""5552""","""9""",['usa'],[],[],[],26-FEB-1987 15:17:11.20,CHAMPION PRODUCTS &lt;CH> APPROVES STOCK SPLIT
5,Computer Terminal Systems Inc said\r\nit has c...,"""NORM""",['acq'],"""TRAIN""","""TRAINING-SET""","""5553""","""10""",['usa'],[],[],[],26-FEB-1987 15:18:06.67,COMPUTER TERMINAL SYSTEMS &lt;CPML> COMPLETES ...
6,"Shr 34 cts vs 1.19 dlrs\r\n Net 807,000 vs ...","""NORM""",['earn'],"""TRAIN""","""TRAINING-SET""","""5554""","""11""",['usa'],[],[],[],26-FEB-1987 15:18:59.34,COBANCO INC &lt;CBCO> YEAR NET
7,"Ohio Mattress Co said its first\r\nquarter, en...","""NORM""",['earn' 'acq'],"""TRAIN""","""TRAINING-SET""","""5555""","""12""",['usa'],[],[],[],26-FEB-1987 15:19:15.45,OHIO MATTRESS &lt;OMT> MAY HAVE LOWER 1ST QTR NET
8,Oper shr loss two cts vs profit seven cts\r\n ...,"""NORM""",['earn'],"""TRAIN""","""TRAINING-SET""","""5556""","""13""",['usa'],[],[],[],26-FEB-1987 15:20:13.09,AM INTERNATIONAL INC &lt;AM> 2ND QTR JAN 31
9,Shr one dlr vs 73 cts\r\n Net 12.6 mln vs 1...,"""NORM""",['earn'],"""TRAIN""","""TRAINING-SET""","""5557""","""14""",['usa'],[],[],[],26-FEB-1987 15:20:27.17,BROWN-FORMAN INC &lt;BFD> 4TH QTR NET


## Sección 4: Limpieza de Datos

In [10]:
def clean_dataset(df):
    """
    Limpia el dataset: rellena valores nulos, limpia columnas y crea campo de documento.

    Args:
        df (pd.DataFrame): DataFrame a limpiar

    Returns:
        pd.DataFrame: DataFrame limpio
    """
    # Copia para no modificar el original
    df = df.copy()

    # Completar valores nulos importantes
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")

    # Limpiar columnas innecesarias
    cols = ["text_type", "lewis_split", "cgis_split"]

    for col in cols:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace('"', '', regex=False)
        )

    # Crear documentos combinando título y texto
    df["document"] = (
        df["title"].astype(str)
        + " " +
        df["text"].astype(str)
    )

    # Eliminar solo documentos totalmente vacíos
    df = df[
        df["document"].str.strip() != ""
    ]

    print(f"Nuevo tamaño del dataset: {df.shape}")

    return df

In [11]:
corpus = clean_dataset(corpus)


Nuevo tamaño del dataset: (9603, 14)


In [12]:
print("Dataset limpio - Primeras filas:")
corpus.head(10)

Dataset limpio - Primeras filas:


,text,text_type,topics,lewis_split,cgis_split,old_id,new_id,places,people,orgs,exchanges,date,title,document
0,Showers continued throughout the week in\r\nth...,NORM,['cocoa'],TRAIN,TRAINING-SET,"""5544""","""1""",['el-salvador' 'usa' 'uruguay'],[],[],[],26-FEB-1987 15:01:01.79,BAHIA COCOA REVIEW,BAHIA COCOA REVIEW Showers continued throughou...
1,The U.S. Agriculture Department\r\nreported th...,NORM,['grain' 'wheat' 'corn' 'barley' 'oat' 'sorghum'],TRAIN,TRAINING-SET,"""5548""","""5""",['usa'],[],[],[],26-FEB-1987 15:10:44.60,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESERVE,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESER...
2,Argentine grain board figures show\r\ncrop reg...,NORM,['veg-oil' 'linseed' 'lin-oil' 'soy-oil' 'sun-...,TRAIN,TRAINING-SET,"""5549""","""6""",['argentina'],[],[],[],26-FEB-1987 15:14:36.41,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS ...
3,Moody's Investors Service Inc said it\r\nlower...,NORM,[],TRAIN,TRAINING-SET,"""5551""","""8""",['usa'],[],[],[],26-FEB-1987 15:15:40.12,USX &lt;X> DEBT DOWGRADED BY MOODY'S,USX &lt;X> DEBT DOWGRADED BY MOODY'S Moody's I...
4,Champion Products Inc said its\r\nboard of dir...,NORM,['earn'],TRAIN,TRAINING-SET,"""5552""","""9""",['usa'],[],[],[],26-FEB-1987 15:17:11.20,CHAMPION PRODUCTS &lt;CH> APPROVES STOCK SPLIT,CHAMPION PRODUCTS &lt;CH> APPROVES STOCK SPLIT...
5,Computer Terminal Systems Inc said\r\nit has c...,NORM,['acq'],TRAIN,TRAINING-SET,"""5553""","""10""",['usa'],[],[],[],26-FEB-1987 15:18:06.67,COMPUTER TERMINAL SYSTEMS &lt;CPML> COMPLETES ...,COMPUTER TERMINAL SYSTEMS &lt;CPML> COMPLETES ...
6,"Shr 34 cts vs 1.19 dlrs\r\n Net 807,000 vs ...",NORM,['earn'],TRAIN,TRAINING-SET,"""5554""","""11""",['usa'],[],[],[],26-FEB-1987 15:18:59.34,COBANCO INC &lt;CBCO> YEAR NET,COBANCO INC &lt;CBCO> YEAR NET Shr 34 cts vs 1...
7,"Ohio Mattress Co said its first\r\nquarter, en...",NORM,['earn' 'acq'],TRAIN,TRAINING-SET,"""5555""","""12""",['usa'],[],[],[],26-FEB-1987 15:19:15.45,OHIO MATTRESS &lt;OMT> MAY HAVE LOWER 1ST QTR NET,OHIO MATTRESS &lt;OMT> MAY HAVE LOWER 1ST QTR ...
8,Oper shr loss two cts vs profit seven cts\r\n ...,NORM,['earn'],TRAIN,TRAINING-SET,"""5556""","""13""",['usa'],[],[],[],26-FEB-1987 15:20:13.09,AM INTERNATIONAL INC &lt;AM> 2ND QTR JAN 31,AM INTERNATIONAL INC &lt;AM> 2ND QTR JAN 31 Op...
9,Shr one dlr vs 73 cts\r\n Net 12.6 mln vs 1...,NORM,['earn'],TRAIN,TRAINING-SET,"""5557""","""14""",['usa'],[],[],[],26-FEB-1987 15:20:27.17,BROWN-FORMAN INC &lt;BFD> 4TH QTR NET,BROWN-FORMAN INC &lt;BFD> 4TH QTR NET Shr one ...


### Sección 5: Pipeline de Preprocesamiento

El preprocesamiento incluye:
1. **Tokenización**: Dividir documentos en palabras
2. **Eliminación de stopwords**: Remover palabras comunes sin valor semántico
3. **Normalización**: Convertir a minúsculas, quitar acentos, eliminar caracteres especiales
4. **Stemming**: Reducir palabras a su raíz

#### 5.1 Tokenización

In [13]:
def tokenize(corpus):
    """
    Tokeniza cada documento en el corpus en palabras individuales.

    Args:
        corpus (pd.DataFrame): DataFrame con documentos

    Returns:
        pd.DataFrame: DataFrame con columna 'tokens'
    """
    # Tokenizar cada documento
    corpus["tokens"] = (corpus["document"].apply(word_tokenize))

    print(f"Tokens creados. Ejemplo (primeros 10): {corpus['tokens'].iloc[0][:10]}")
    return corpus


#### 5.2 Eliminación de Stopwords

In [14]:
def quit_stopwords(corpus):
    """
    Elimina stopwords (palabras comunes) del corpus y convierte a minúsculas.

    Args:
        corpus (pd.DataFrame): DataFrame con tokens

    Returns:
        pd.DataFrame: DataFrame con columna 'no_stopwords'
    """
    stop_words = set(stopwords.words("english"))

    # Poner en minúsculas
    corpus["no_stopwords"] = (
        corpus["tokens"]
        .apply(
            lambda words: [
                word.lower()
                for word in words
            ]
        )
    )

    # Quitar las stopwords
    corpus["no_stopwords"] = (
        corpus["no_stopwords"]
        .apply(
            lambda words: [
                word
                for word in words
                if word not in stop_words
            ]
        )
    )

    print(f"Stopwords removidos. Ejemplo: {corpus['no_stopwords'].iloc[0][:10]}")
    return corpus


#### 5.3 Normalización

In [15]:
def normalize(tokens):
    """
    Normaliza tokens: convierte a minúsculas, elimina acentos,
    y mantiene solo caracteres alfabéticos.

    Args:
        tokens (list): Lista de tokens

    Returns:
        list: Lista de tokens normalizados
    """
    normalized = []

    for token in tokens:
        # lowercase (case folding)
        token = token.lower()

        # quitar acentos
        token = unicodedata.normalize(
            "NFKD",
            token
        ).encode(
            "ascii",
            "ignore"
        ).decode("utf-8")

        # dejar solo letras
        token = re.sub(
            r"[^a-z]",
            "",
            token
        )

        # evitar vacíos
        if token:
            normalized.append(token)

    return normalized


#### 5.4 Stemming

In [16]:
def stemming(corpus):
    """
    Aplica stemming (reducción a raíz) a los tokens normalizados.

    Args:
        corpus (pd.DataFrame): DataFrame con columna 'normalized'

    Returns:
        pd.DataFrame: DataFrame con columna 'stemmed'
    """
    stemmer = PorterStemmer()

    corpus["stemmed"] = (
        corpus["normalized"]
        .apply(
            lambda words: [
                stemmer.stem(word)
                for word in words
            ]
        )
    )

    print(f"Stemming aplicado. Ejemplo: {corpus['stemmed'].iloc[0][:10]}")
    return corpus

#### 5.5 Pipeline Completo de Preprocesamiento

In [17]:
def preprocessing(corpus):
    """
    Ejecuta el pipeline completo de preprocesamiento:
    tokenización → eliminación de stopwords → normalización → stemming

    Args:
        corpus (pd.DataFrame): DataFrame con documentos originales

    Returns:
        pd.DataFrame: DataFrame preprocesado con columnas de tokens, stopwords, normalized, y stemmed
    """
    print("Iniciando pipeline de preprocesamiento...\n")

    # Tokenizar
    print("1. Tokenización...")
    corpus = tokenize(corpus)

    # Quitar stopwords
    print("\n2. Eliminación de stopwords...")
    corpus = quit_stopwords(corpus)

    # Normalización (case folding, acentos, etc.)
    print("\n3. Normalización...")
    corpus["normalized"] = (corpus["no_stopwords"].apply(normalize))

    # Stemming
    print("\n4. Stemming...")
    corpus = stemming(corpus)

    print("\nPipeline completado exitosamente")
    return corpus


### Sección 6: Ejecución del Pipeline

In [18]:
print("Procesando corpus...")
corpus = preprocessing(corpus)

Procesando corpus...
Iniciando pipeline de preprocesamiento...

1. Tokenización...
Tokens creados. Ejemplo (primeros 10): ['BAHIA', 'COCOA', 'REVIEW', 'Showers', 'continued', 'throughout', 'the', 'week', 'in', 'the']

2. Eliminación de stopwords...
Stopwords removidos. Ejemplo: ['bahia', 'cocoa', 'review', 'showers', 'continued', 'throughout', 'week', 'bahia', 'cocoa', 'zone']

3. Normalización...

4. Stemming...
Stemming aplicado. Ejemplo: ['bahia', 'cocoa', 'review', 'shower', 'continu', 'throughout', 'week', 'bahia', 'cocoa', 'zone']

✓ Pipeline completado exitosamente


### Sección 7: Resultados

In [19]:
print("Corpus preprocesado - Información final:")
print(f"Tamaño del corpus: {corpus.shape}")
print(f"\nColumnas disponibles: {list(corpus.columns)}")

Corpus preprocesado - Información final:
Tamaño del corpus: (9603, 18)

Columnas disponibles: ['text', 'text_type', 'topics', 'lewis_split', 'cgis_split', 'old_id', 'new_id', 'places', 'people', 'orgs', 'exchanges', 'date', 'title', 'document', 'tokens', 'no_stopwords', 'normalized', 'stemmed']


In [20]:
print("Ejemplo de tokens procesados (primeros 100 elementos del primer documento):")
print(corpus["stemmed"].iloc[0][:100])

Ejemplo de tokens procesados (primeros 100 elementos del primer documento):
['bahia', 'cocoa', 'review', 'shower', 'continu', 'throughout', 'week', 'bahia', 'cocoa', 'zone', 'allevi', 'drought', 'sinc', 'earli', 'januari', 'improv', 'prospect', 'come', 'temporao', 'although', 'normal', 'humid', 'level', 'restor', 'comissaria', 'smith', 'said', 'weekli', 'review', 'dri', 'period', 'mean', 'temporao', 'late', 'year', 'arriv', 'week', 'end', 'februari', 'bag', 'kilo', 'make', 'cumul', 'total', 'season', 'mln', 'stage', 'last', 'year', 'seem', 'cocoa', 'deliv', 'earlier', 'consign', 'includ', 'arriv', 'figur', 'comissaria', 'smith', 'said', 'still', 'doubt', 'much', 'old', 'crop', 'cocoa', 'still', 'avail', 'harvest', 'practic', 'come', 'end', 'total', 'bahia', 'crop', 'estim', 'around', 'mln', 'bag', 'sale', 'stand', 'almost', 'mln', 'hundr', 'thousand', 'bag', 'still', 'hand', 'farmer', 'middlemen', 'export', 'processor', 'doubt', 'much', 'cocoa', 'would', 'fit', 'export', 'shipper', 'ex

In [21]:
print("\nEstadísticas de tokenización y preprocesamiento:")
print(f"Promedio de tokens por documento: {corpus['tokens'].apply(len).mean():.2f}")
print(f"Promedio de tokens sin stopwords: {corpus['no_stopwords'].apply(len).mean():.2f}")
print(f"Promedio de tokens finales (después de stemming): {corpus['stemmed'].apply(len).mean():.2f}")



Estadísticas de tokenización y preprocesamiento:
Promedio de tokens por documento: 149.23
Promedio de tokens sin stopwords: 106.29
Promedio de tokens finales (después de stemming): 84.20


In [28]:
corpus[["document", "stemmed"]].head(5)

,document,stemmed
0,BAHIA COCOA REVIEW Showers continued throughou...,"[bahia, cocoa, review, shower, continu, throug..."
1,NATIONAL AVERAGE PRICES FOR FARMER-OWNED RESER...,"[nation, averag, price, farmerown, reserv, us,..."
2,ARGENTINE 1986/87 GRAIN/OILSEED REGISTRATIONS ...,"[argentin, grainoilse, registr, argentin, grai..."
3,USX &lt;X> DEBT DOWGRADED BY MOODY'S Moody's I...,"[usx, lt, x, debt, dowgrad, moodi, s, moodi, s..."
4,CHAMPION PRODUCTS &lt;CH> APPROVES STOCK SPLIT...,"[champion, product, lt, ch, approv, stock, spl..."


## Construcción del índice

• Construcción de un índice invertido que almacene, para cada término, los documentos en los que
aparece y su frecuencia

In [29]:
def create_inverted_index(corpus):
    inverted_index = {}

    for _, rows in corpus.iterrows():
        id_doc = rows["new_id"]
        tokens = rows["stemmed"]

        for token in tokens:

            if token not in inverted_index:
                inverted_index[token] = {}

            if id_doc not in inverted_index[token]:
                inverted_index[token][id_doc] = 1
            else:
                inverted_index[token][id_doc] += 1

    return inverted_index

In [30]:
inverted_index = create_inverted_index(corpus)

In [31]:
print(f"Indice invertido de market:{inverted_index["switzerland"]}")
# print(inverted_index["stocks"])
# print(inverted_index["oil"])

Indice invertido de market:{'"10"': 1, '"179"': 3, '"204"': 1, '"359"': 1, '"394"': 1, '"415"': 1, '"419"': 1, '"427"': 1, '"458"': 1, '"468"': 1, '"501"': 2, '"545"': 1, '"759"': 1, '"797"': 1, '"1248"': 1, '"1675"': 1, '"2214"': 1, '"2257"': 1, '"2280"': 2, '"2503"': 1, '"2790"': 1, '"3192"': 1, '"3536"': 1, '"3554"': 1, '"3792"': 1, '"3844"': 1, '"3901"': 1, '"3923"': 1, '"4022"': 1, '"4638"': 1, '"4691"': 1, '"4692"': 1, '"5032"': 1, '"5813"': 1, '"5824"': 1, '"5855"': 1, '"6658"': 2, '"7035"': 1, '"7043"': 1, '"7088"': 4, '"7104"': 1, '"7175"': 1, '"7311"': 1, '"7950"': 1, '"8199"': 1, '"8421"': 1, '"8635"': 1, '"8667"': 1, '"8675"': 1, '"8855"': 1, '"9325"': 1, '"9410"': 1, '"9448"': 1, '"9570"': 1, '"9835"': 1, '"9866"': 1, '"9905"': 1, '"10428"': 1, '"10849"': 1, '"11163"': 1, '"11830"': 2, '"12136"': 1, '"12482"': 1, '"13005"': 1}
